# Chapter 08: Production Retrieval-Augmented Generation (RAG)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abda-dc/ai-engineering-zero-to-hero/blob/main/book/chapters/part3-rag-data/ch08-production-rag-systems.ipynb)

**Part:** Part III: Vector Databases, Data & Production RAG  
**Chapter Identifier:** `ch08`  

## Core Concepts Covered in this Chapter:
- **Why RAG**
- **Architecture**
- **Document Loading**
- **Chunking Strategies**
- **Embedding Models**
- **Indexing**
- **Top-k**
- **Hybrid BM25 Search**
- **Cross-Encoder Re-ranking**
- **Query Rewriting**
- **Prompt Augmentation**
- **Context Mgmt**
- **Citations/Grounding**
- **Multi-hop RAG**
- **GraphRAG**
- **Agentic RAG**
- **RAG Evals**

---


## 1. Why RAG & Production System Architecture

LLMs suffer from knowledge cutoffs, domain gaps, and hallucinations. Retrieval-Augmented Generation grounds generations in fresh, proprietary external knowledge without expensive model re-training.

```mermaid
sequenceDiagram
    autonumber
    actor User
    participant Rewriter as Query Rewriter
    participant Hybrid as Hybrid Search (BM25 + Dense)
    participant Rerank as Cross-Encoder Reranker
    participant LLM as Aligned LLM

    User->>Rewriter: "How do I setup MCP with Docker?"
    Rewriter->>Hybrid: Expanded Query & Keywords
    Hybrid->>Rerank: Top-25 Candidate Passages
    Rerank->>LLM: Top-5 High-Precision Passages + Prompt
    LLM-->>User: Grounded Answer with Bracketed Citations [1][2]
```


In [ ]:
print("Production RAG Architecture: Dual-stage retrieval + Cross-encoder reranking + Grounding verification.")

## 2. Chunking Strategies: Fixed, Sliding Window & Semantic

Chunking splits documents into digestible passages. Fixed-size chunking can split sentences mid-thought. Sliding window adds overlap to preserve boundary context. Semantic chunking splits based on semantic shifts between sentences.


In [ ]:
def chunk_text_sliding_window(text: str, chunk_size: int = 15, overlap: int = 5) -> list:
    words = text.split()
    chunks = []
    step = chunk_size - overlap
    for i in range(0, len(words), step):
        chunk = " ".join(words[i:i + chunk_size])
        chunks.append(chunk)
        if i + chunk_size >= len(words):
            break
    return chunks

doc = "Retrieval-Augmented Generation connects foundation models to external verified knowledge bases, eliminating hallucinations and enabling enterprise data discovery across systems."
chunks = chunk_text_sliding_window(doc, chunk_size=8, overlap=3)
for idx, c in enumerate(chunks, 1):
    print(f"Chunk {idx}: '{c}'")

## 3. Query Rewriting & Cross-Encoder Re-ranking

Raw user queries are often ambiguous or underspecified. Query rewriting expands acronyms and generates hypothetical document embeddings (HyDE). Cross-encoders then perform full cross-attention over (query, passage) pairs to re-order the top candidates.


In [ ]:
def simulate_cross_encoder_rerank(query: str, passages: list) -> list:
    scored = []
    q_words = set(query.lower().split())
    for p in passages:
        p_words = set(p.lower().split())
        overlap = len(q_words.intersection(p_words))
        score = overlap / max(len(q_words), 1)
        scored.append({"passage": p, "relevance_score": round(score, 3)})
    scored.sort(key=lambda x: x["relevance_score"], reverse=True)
    return scored

candidates = [
    "Vector search enables rapid similarity retrieval.",
    "RAG systems ground LLM responses with retrieved enterprise documents.",
    "Baking sourdough bread requires flour, water, and levain."
]
reranked = simulate_cross_encoder_rerank("What is RAG retrieval?", candidates)
print("Reranked Passages:")
for r in reranked:
    print(f"  Score: {r['relevance_score']} -> {r['passage']}")

---
## Hands-On Capstone Lab: Build an Advanced RAG Pipeline with Semantic Chunking, Reranking & Citations

### Objective:
Build a modular RAG pipeline that processes documents, executes retrieval, reranks candidates, and constructs a grounded response with source citations.

### Instructions:
1. Ingest document and chunk into passages.
2. Execute retrieval and reranking for a query.
3. Format context with `[Doc 1]`, `[Doc 2]` citations into the final generation prompt.


In [ ]:
class AdvancedRAGPipeline:
    def __init__(self, raw_document: str):
        self.passages = chunk_text_sliding_window(raw_document, chunk_size=10, overlap=3)

    def answer_query(self, query: str) -> dict:
        reranked = simulate_cross_encoder_rerank(query, self.passages)
        top_k = reranked[:2]
        simulated_answer = (
            f"Based on the verified context, {top_k[0]['passage']} [1]. "
            f"Furthermore, {top_k[1]['passage']} [2]."
        )
        return {
            "query": query,
            "citations": top_k,
            "synthesized_answer": simulated_answer
        }

knowledge_base = "Model Context Protocol connects AI assistants to local data sources and tools securely. MCP utilizes JSON-RPC communication between hosts and servers. Standardized tool schemas prevent custom API integrations."
rag = AdvancedRAGPipeline(knowledge_base)
response = rag.answer_query("How does MCP communicate?")
print("Synthesized Grounded Answer:\n", response["synthesized_answer"])

---
## Chapter Mastery Self-Check Quiz

Test your understanding of the concepts covered in this chapter.

### Question 1: What is the primary advantage of a Cross-Encoder reranker compared to bi-encoder vector similarity?
- **A)** Cross-encoders are 10,000x faster than bi-encoders
- **B)** Cross-encoders perform full joint cross-attention between every query token and passage token, capturing complex semantic interactions
- **C)** Cross-encoders do not require any GPU compute
- **D)** Cross-encoders eliminate the need for vector databases

<details>
<summary><b>View Answer & Explanation</b></summary>

**Correct Answer:** B) Cross-encoders perform full joint cross-attention between every query token and passage token, capturing complex semantic interactions

**Explanation:** Bi-encoders encode queries and passages independently into single vectors. Cross-encoders encode them jointly, capturing subtle token interactions at the cost of higher latency.
</details>

### Question 2: In RAG evaluations (e.g. Ragas), what does 'Faithfulness' measure?
- **A)** How quickly the model responds
- **B)** Whether all claims in the generated response are strictly grounded in and inferable from the retrieved context
- **C)** Whether the user liked the answer
- **D)** How many tokens were generated

<details>
<summary><b>View Answer & Explanation</b></summary>

**Correct Answer:** B) Whether all claims in the generated response are strictly grounded in and inferable from the retrieved context

**Explanation:** Faithfulness evaluates whether the generated output is factually consistent with the retrieved context (i.e. zero hallucinations beyond provided text).
</details>

### Question 3: What is 'GraphRAG'?
- **A)** RAG using only pie charts
- **B)** Combining Knowledge Graphs (entities, relationships) with vector search to answer global multi-hop summary questions
- **C)** A tool that draws network cables
- **D)** An image generation library

<details>
<summary><b>View Answer & Explanation</b></summary>

**Correct Answer:** B) Combining Knowledge Graphs (entities, relationships) with vector search to answer global multi-hop summary questions

**Explanation:** GraphRAG extracts knowledge graphs (communities of entities and relationships) to summarize themes across large corpora that standard vector chunks miss.
</details>

### Question 4: Why is sliding-window chunking with overlap preferred over strict non-overlapping chunking?
- **A)** It ensures that semantic meaning and entity relationships that span across chunk boundaries are not severed
- **B)** It cuts database storage in half
- **C)** It eliminates the need for tokenizers
- **D)** It enables models to output audio

<details>
<summary><b>View Answer & Explanation</b></summary>

**Correct Answer:** A) It ensures that semantic meaning and entity relationships that span across chunk boundaries are not severed

**Explanation:** Overlapping chunk boundaries prevent splitting critical statements or sentence transitions mid-thought, preserving contextual coherence.
</details>
